# Calib-BERT JUCS — Canonical Evidence-First Experiment

This notebook replaces the earlier experimental pipeline. Its purpose is to produce a **single auditable set of results** without test-set-driven model selection.

## Locked protocol

1. The official Dreaddit training split is partitioned **by `post_id`** into:
   - model-fit subset (approximately 70%),
   - validation subset (approximately 15%),
   - calibration subset (approximately 15%).
2. Candidate calibration strategies are selected **only from the validation subset**.
3. The selected configuration and the Vanilla BERT baseline are retrained on model-fit + validation data.
4. Temperature scaling is fitted **only on the untouched calibration subset**.
5. The official 715-instance test split is loaded only after the selected configuration has been written to disk.
6. Official test results are never used to change epsilon, the selected model, the temperature, or any other hyperparameter.
7. Topic diagnostics are post-prediction analyses and do not affect classification training or model selection.
8. Every output is saved with input hashes, split assignments, software versions, and a run manifest.

**Do not edit the protocol after viewing official test results.** If the protocol must change, start a new run directory and report the change transparently.

In [ ]:
# ============================================================
# 0. Optional installation (Google Colab)
# ============================================================
# Run once in a fresh Colab runtime. The exact installed versions
# are recorded later in pip_freeze.txt and environment.json.
!pip install -q transformers accelerate scikit-learn pandas numpy matplotlib tqdm scipy statsmodels
!pip install -q top2vec umap-learn hdbscan gensim sentence-transformers

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
from pathlib import Path

RUN_NAME = "calibbert_jucs_canonical_v1"
ROOT = Path.cwd()
OUT_DIR = ROOT / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = ROOT / "dreaddit-train.csv"
TEST_PATH = ROOT / "dreaddit-test.csv"

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 3
NUM_LABELS = 2
WARMUP_STEPS = 0
GRAD_CLIP = 1.0

SPLIT_SEED = 42
PRIMARY_SEED = 42
REPEATED_SEEDS = [13, 21, 42, 87, 100]

CANDIDATES = [
    {"key": "M0_Vanilla_BERT", "mode": "none", "epsilon": 0.00},
    {"key": "M1_LS_0.05", "mode": "ls", "epsilon": 0.05},
    {"key": "M1_LS_0.10", "mode": "ls", "epsilon": 0.10},
    {"key": "M1_LS_0.15", "mode": "ls", "epsilon": 0.15},
    {"key": "M2_AALS_0.10", "mode": "aals", "epsilon": 0.10},
    {"key": "M2_AALS_0.15", "mode": "aals", "epsilon": 0.15},
]

# Predeclared selection rule: lowest validation ECE, then lowest
# validation Brier Score, then highest validation F1-score.
SELECTION_RULE = ["ece_ascending", "brier_ascending", "f1_descending"]

RUN_ALL_FINAL_CANDIDATES = True
RUN_REPEATED_SEEDS = True
RUN_TOP2VEC = True

BOOTSTRAP_ITERATIONS = 10000
TOPIC_BOOTSTRAP_ITERATIONS = 5000

# The official test set must not be used before the selection file exists.
SELECTION_FILE = OUT_DIR / "selected_configuration.json"
TEST_LOCK_FILE = OUT_DIR / "OFFICIAL_TEST_EVALUATION_LOCK.json"

In [ ]:
# ============================================================
# 2. Imports, determinism, hardware, and software provenance
# ============================================================
import os, sys, json, math, time, random, hashlib, platform, subprocess, warnings, shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    log_loss,
)
from scipy.stats import binomtest
from statsmodels.stats.contingency_tables import mcnemar
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

def set_all_seeds(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_all_seeds(PRIMARY_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type != "cuda":
    raise RuntimeError(
        "GPU CUDA tidak terdeteksi. Jalankan notebook ini di Google Colab dengan GPU "
        "(Runtime > Change runtime type > T4 GPU atau yang setara)."
    )

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

environment = {
    "run_name": RUN_NAME,
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
}
try:
    import transformers, sklearn, scipy, statsmodels
    environment.update({
        "transformers": transformers.__version__,
        "scikit_learn": sklearn.__version__,
        "scipy": scipy.__version__,
        "statsmodels": statsmodels.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    })
except Exception:
    pass

with open(OUT_DIR / "environment.json", "w", encoding="utf-8") as f:
    json.dump(environment, f, indent=2)

freeze = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True
)
(OUT_DIR / "pip_freeze.txt").write_text(freeze, encoding="utf-8")
environment

## Stage A — development data only

The cells in this stage load **only the official training split**. The official test file is not read until the selected configuration has been fixed and saved.

In [ ]:
# ============================================================
# 3. Load and audit official training data only
# ============================================================
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Missing: {TRAIN_PATH}")
if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing: {TEST_PATH}. The file is only hashed here and loaded after selection."
    )

train_full = pd.read_csv(TRAIN_PATH)

required = {"text", "label", "confidence", "id", "post_id"}
missing = required.difference(train_full.columns)
if missing:
    raise ValueError(f"Missing required train columns: {sorted(missing)}")

train_full = train_full.copy()
train_full["text"] = train_full["text"].astype(str)
train_full["label"] = train_full["label"].astype(int)
train_full["confidence"] = pd.to_numeric(
    train_full["confidence"], errors="raise"
).astype(float)
train_full["post_id"] = train_full["post_id"].astype(str)

if set(train_full["label"].unique()) != {0, 1}:
    raise ValueError("Labels must be exactly {0,1}.")
if train_full[["text", "label", "confidence", "id", "post_id"]].isna().any().any():
    raise ValueError("Null values found in required training columns.")
if not train_full["confidence"].between(0, 1).all():
    raise ValueError("Annotator-agreement values must be in [0,1].")

input_hashes = {
    "dreaddit_train_sha256": sha256_file(TRAIN_PATH),
    "dreaddit_test_sha256": sha256_file(TEST_PATH),
}

train_audit = {
    "rows": int(len(train_full)),
    "columns": int(len(train_full.columns)),
    "unique_posts": int(train_full["post_id"].nunique()),
    "label_0": int((train_full["label"] == 0).sum()),
    "label_1": int((train_full["label"] == 1).sum()),
    "agreement_mean": float(train_full["confidence"].mean()),
    "agreement_min": float(train_full["confidence"].min()),
    "agreement_max": float(train_full["confidence"].max()),
    **input_hashes,
}
with open(OUT_DIR / "official_train_audit.json", "w", encoding="utf-8") as f:
    json.dump(train_audit, f, indent=2)

train_audit

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# 4. Post-grouped 70/15/15 development split
# ============================================================
# Twenty stratified group folds are formed by post_id:
# folds 0-13 = fit, 14-16 = validation, 17-19 = calibration.
# This gives approximately 70/15/15 while ensuring that segments
# from the same Reddit post never cross internal subsets.

groups = train_full["post_id"].astype(str).values
y = train_full["label"].values
fold_id = np.empty(len(train_full), dtype=int)

sgkf = StratifiedGroupKFold(
    n_splits=20,
    shuffle=True,
    random_state=SPLIT_SEED,
)
for fold, (_, held_idx) in enumerate(
    sgkf.split(np.zeros(len(train_full)), y, groups=groups)
):
    fold_id[held_idx] = fold

split_name = np.where(
    fold_id < 14,
    "fit",
    np.where(fold_id < 17, "validation", "calibration"),
)

split_assignment = pd.DataFrame({
    "row_index": np.arange(len(train_full)),
    "id": train_full["id"].values,
    "post_id": train_full["post_id"].values,
    "label": train_full["label"].values,
    "confidence": train_full["confidence"].values,
    "fold_id": fold_id,
    "split": split_name,
})
split_assignment.to_csv(
    OUT_DIR / "development_split_assignment.csv", index=False
)

# Hard leakage checks.
for a, b in [
    ("fit", "validation"),
    ("fit", "calibration"),
    ("validation", "calibration"),
]:
    pa = set(split_assignment.loc[split_assignment["split"] == a, "post_id"])
    pb = set(split_assignment.loc[split_assignment["split"] == b, "post_id"])
    overlap = pa.intersection(pb)
    assert len(overlap) == 0, f"post_id leakage between {a} and {b}"

fit_df = train_full.iloc[np.where(split_name == "fit")[0]].reset_index(drop=True)
validation_df = train_full.iloc[
    np.where(split_name == "validation")[0]
].reset_index(drop=True)
calibration_df = train_full.iloc[
    np.where(split_name == "calibration")[0]
].reset_index(drop=True)

split_summary_rows = []
for name, df in [
    ("fit", fit_df),
    ("validation", validation_df),
    ("calibration", calibration_df),
]:
    split_summary_rows.append({
        "split": name,
        "n": len(df),
        "unique_posts": df["post_id"].nunique(),
        "label_0": int((df["label"] == 0).sum()),
        "label_1": int((df["label"] == 1).sum()),
        "agreement_mean": df["confidence"].mean(),
        "agreement_min": df["confidence"].min(),
        "agreement_max": df["confidence"].max(),
    })

split_summary = pd.DataFrame(split_summary_rows)
split_summary.to_csv(OUT_DIR / "development_split_summary.csv", index=False)
split_summary

In [ ]:
# ============================================================
# 5. Tokenizer, dataset, and deterministic loaders
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DreadditDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True).copy()
        enc = tokenizer(
            self.df["text"].astype(str).tolist(),
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = torch.tensor(
            self.df["label"].astype(int).values, dtype=torch.long
        )
        self.agreements = torch.tensor(
            self.df["confidence"].astype(float).values, dtype=torch.float32
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
            "agreement": self.agreements[idx],
            "row_index": idx,
        }

def make_loader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool,
    seed: int,
) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=True,
        generator=generator,
    )

fit_ds = DreadditDataset(fit_df)
validation_ds = DreadditDataset(validation_df)
calibration_ds = DreadditDataset(calibration_df)

In [ ]:
# ============================================================
# 6. Losses and metrics
# ============================================================
def soft_targets(
    labels,
    agreements=None,
    mode="none",
    epsilon=0.0,
    a_min=None,
    num_labels=2,
):
    if mode == "none" or epsilon <= 0:
        return None

    batch = labels.size(0)
    device = labels.device

    if mode == "ls":
        eps = torch.full((batch,), float(epsilon), device=device)

    elif mode == "aals":
        if agreements is None:
            raise ValueError("AALS requires annotator agreement.")
        if a_min is None:
            raise ValueError("AALS requires a_min from the model-fit subset.")
        agreement = agreements.to(device).float().clamp(float(a_min), 1.0)
        denominator = max(1e-8, 1.0 - float(a_min))
        eps = float(epsilon) * (1.0 - agreement) / denominator
        eps = eps.clamp(0.0, float(epsilon))

    else:
        raise ValueError(f"Unknown mode: {mode}")

    target = torch.full(
        (batch, num_labels), 0.0, dtype=torch.float32, device=device
    )
    target += (eps / (num_labels - 1)).unsqueeze(1)
    target.scatter_(1, labels.unsqueeze(1), (1.0 - eps).unsqueeze(1))
    return target

def loss_function(
    logits,
    labels,
    agreements,
    mode,
    epsilon,
    a_min,
):
    if mode == "none":
        return F.cross_entropy(logits, labels)
    targets = soft_targets(
        labels,
        agreements=agreements,
        mode=mode,
        epsilon=epsilon,
        a_min=a_min,
        num_labels=NUM_LABELS,
    )
    return -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

def expected_calibration_error(y_true, y_pred, confidence, n_bins=10):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    confidence = np.asarray(confidence, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)

    rows, ece = [], 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        if b == n_bins - 1:
            mask = (confidence >= lo) & (confidence <= hi)
        else:
            mask = (confidence >= lo) & (confidence < hi)
        n = int(mask.sum())
        if n == 0:
            rows.append({
                "bin": b + 1,
                "lower": lo,
                "upper": hi,
                "count": 0,
                "mean_confidence": np.nan,
                "accuracy": np.nan,
                "absolute_gap": np.nan,
                "contribution": 0.0,
            })
            continue
        acc = float((y_true[mask] == y_pred[mask]).mean())
        conf = float(confidence[mask].mean())
        gap = abs(acc - conf)
        contribution = (n / len(y_true)) * gap
        ece += contribution
        rows.append({
            "bin": b + 1,
            "lower": lo,
            "upper": hi,
            "count": n,
            "mean_confidence": conf,
            "accuracy": acc,
            "absolute_gap": gap,
            "contribution": contribution,
        })
    return float(ece), pd.DataFrame(rows)

def adaptive_calibration_error(y_true, y_pred, confidence, n_bins=10):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    confidence = np.asarray(confidence, dtype=float)
    order = np.argsort(confidence)
    index_bins = np.array_split(order, n_bins)

    rows, ace = [], 0.0
    for b, idx in enumerate(index_bins, start=1):
        if len(idx) == 0:
            continue
        acc = float((y_true[idx] == y_pred[idx]).mean())
        conf = float(confidence[idx].mean())
        gap = abs(acc - conf)
        contribution = (len(idx) / len(y_true)) * gap
        ace += contribution
        rows.append({
            "adaptive_bin": b,
            "count": len(idx),
            "min_confidence": float(confidence[idx].min()),
            "max_confidence": float(confidence[idx].max()),
            "mean_confidence": conf,
            "accuracy": acc,
            "absolute_gap": gap,
            "contribution": contribution,
        })
    return float(ace), pd.DataFrame(rows)

def metric_bundle(y_true, probabilities, model_key):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    y_pred = probabilities.argmax(axis=1)
    confidence = probabilities.max(axis=1)
    p_stress = probabilities[:, 1]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=1, zero_division=0
    )
    ece, ece_bins = expected_calibration_error(
        y_true, y_pred, confidence, n_bins=10
    )
    ace, ace_bins = adaptive_calibration_error(
        y_true, y_pred, confidence, n_bins=10
    )
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    metrics = {
        "model": model_key,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "brier_score": float(np.mean((p_stress - y_true) ** 2)),
        "nll": float(log_loss(y_true, probabilities, labels=[0, 1])),
        "ece": float(ece),
        "ace": float(ace),
        "hce_0.90_count": int(
            ((y_true != y_pred) & (confidence >= 0.90)).sum()
        ),
        "coverage_0.90": float((confidence >= 0.90).mean()),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    }

    prediction_df = pd.DataFrame({
        "true_label": y_true,
        "predicted_label": y_pred,
        "prob_not_stress": probabilities[:, 0],
        "prob_stress": probabilities[:, 1],
        "confidence": confidence,
        "correct": y_true == y_pred,
    })
    prediction_df["error_type"] = np.select(
        [
            (prediction_df["true_label"] == 0)
            & (prediction_df["predicted_label"] == 1),
            (prediction_df["true_label"] == 1)
            & (prediction_df["predicted_label"] == 0),
        ],
        ["false_positive", "false_negative"],
        default="correct",
    )
    return metrics, ece_bins, ace_bins, prediction_df

In [ ]:
# ============================================================
# 7. Training and prediction helpers
# ============================================================
A_MIN = float(fit_df["confidence"].min())
print("AALS a_min (model-fit subset):", A_MIN)

def predict_logits(model, dataset, seed=PRIMARY_SEED):
    loader = make_loader(
        dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, seed=seed
    )
    model.eval()
    logits_list, labels_list = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predict", leave=False):
            outputs = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
            )
            logits_list.append(outputs.logits.detach().cpu().numpy())
            labels_list.append(batch["labels"].numpy())
    return np.vstack(logits_list), np.concatenate(labels_list)

def train_model(
    model_key,
    train_dataset,
    evaluation_dataset,
    mode,
    epsilon,
    seed,
):
    # Reset immediately before each model so paired models start from
    # the same pretrained checkpoint initialization and data order.
    set_all_seeds(seed)

    train_loader = make_loader(
        train_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        seed=seed,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=total_steps,
    )

    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_losses = []
        for batch in tqdm(
            train_loader,
            desc=f"{model_key} epoch {epoch}/{EPOCHS}",
            leave=False,
        ):
            optimizer.zero_grad(set_to_none=True)
            outputs = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
            )
            loss = loss_function(
                outputs.logits,
                batch["labels"].to(DEVICE),
                batch["agreement"].to(DEVICE),
                mode=mode,
                epsilon=epsilon,
                a_min=A_MIN,
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), GRAD_CLIP
            )
            optimizer.step()
            scheduler.step()
            epoch_losses.append(float(loss.detach().cpu().item()))

        history.append({
            "model": model_key,
            "seed": seed,
            "epoch": epoch,
            "train_loss": float(np.mean(epoch_losses)),
        })

    logits, labels = predict_logits(model, evaluation_dataset, seed=seed)
    probabilities = torch.softmax(
        torch.tensor(logits, dtype=torch.float32), dim=1
    ).numpy()
    metrics, ece_bins, ace_bins, predictions = metric_bundle(
        labels, probabilities, model_key
    )
    return {
        "model": model,
        "logits": logits,
        "labels": labels,
        "metrics": metrics,
        "ece_bins": ece_bins,
        "ace_bins": ace_bins,
        "predictions": predictions,
        "history": pd.DataFrame(history),
    }

def release_model(model):
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 8. Candidate selection on validation data only
# ============================================================
validation_rows = []
validation_histories = []

for cfg in CANDIDATES:
    result = train_model(
        model_key=cfg["key"],
        train_dataset=fit_ds,
        evaluation_dataset=validation_ds,
        mode=cfg["mode"],
        epsilon=cfg["epsilon"],
        seed=PRIMARY_SEED,
    )

    row = {
        **result["metrics"],
        "mode": cfg["mode"],
        "epsilon": cfg["epsilon"],
        "evaluation_partition": "validation",
    }
    validation_rows.append(row)
    validation_histories.append(result["history"])

    pred = result["predictions"].copy()
    pred.insert(0, "text", validation_df["text"].values)
    pred.to_csv(
        OUT_DIR / f"validation_predictions_{cfg['key']}.csv",
        index=False,
    )
    result["ece_bins"].to_csv(
        OUT_DIR / f"validation_ece_bins_{cfg['key']}.csv",
        index=False,
    )
    result["ace_bins"].to_csv(
        OUT_DIR / f"validation_ace_bins_{cfg['key']}.csv",
        index=False,
    )

    release_model(result["model"])

validation_metrics = pd.DataFrame(validation_rows)
validation_metrics = validation_metrics.sort_values(
    ["ece", "brier_score", "f1_score"],
    ascending=[True, True, False],
).reset_index(drop=True)
validation_metrics["selection_rank"] = np.arange(
    1, len(validation_metrics) + 1
)
validation_metrics.to_csv(
    OUT_DIR / "validation_candidate_metrics.csv", index=False
)
pd.concat(validation_histories, ignore_index=True).to_csv(
    OUT_DIR / "validation_training_history.csv", index=False
)

selected_row = validation_metrics.iloc[0]
selected_cfg = next(
    c for c in CANDIDATES if c["key"] == selected_row["model"]
)
selection_record = {
    "selected_model": selected_cfg["key"],
    "mode": selected_cfg["mode"],
    "epsilon": float(selected_cfg["epsilon"]),
    "selection_partition": "validation",
    "selection_rule": SELECTION_RULE,
    "primary_seed": PRIMARY_SEED,
    "split_seed": SPLIT_SEED,
    "selected_validation_metrics": {
        k: float(selected_row[k])
        for k in [
            "accuracy",
            "precision",
            "recall",
            "f1_score",
            "brier_score",
            "nll",
            "ece",
            "ace",
        ]
    },
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
with open(SELECTION_FILE, "w", encoding="utf-8") as f:
    json.dump(selection_record, f, indent=2)

print("LOCKED SELECTED CONFIGURATION")
display(validation_metrics)
selection_record

## Stage B — official held-out test evaluation

The selected configuration has now been saved. From this point onward, no hyperparameter or model-selection rule may be changed on the basis of official test results.

In [ ]:
# ============================================================
# 9. Load and audit official test data after selection is locked
# ============================================================
if not SELECTION_FILE.exists():
    raise RuntimeError(
        "selected_configuration.json is missing. Do not load the test set "
        "before completing validation-only selection."
    )

test_df = pd.read_csv(TEST_PATH)
missing = required.difference(test_df.columns)
if missing:
    raise ValueError(f"Missing required test columns: {sorted(missing)}")

test_df = test_df.copy()
test_df["text"] = test_df["text"].astype(str)
test_df["label"] = test_df["label"].astype(int)
test_df["confidence"] = pd.to_numeric(
    test_df["confidence"], errors="raise"
).astype(float)
test_df["post_id"] = test_df["post_id"].astype(str)

assert sha256_file(TEST_PATH) == input_hashes["dreaddit_test_sha256"]

if test_df[["text", "label", "confidence", "id", "post_id"]].isna().any().any():
    raise ValueError("Null values found in required test columns.")
if set(test_df["label"].unique()) != {0, 1}:
    raise ValueError("Test labels must be exactly {0,1}.")

official_overlap_posts = set(train_full["post_id"]).intersection(
    set(test_df["post_id"])
)
official_overlap_ids = set(train_full["id"]).intersection(set(test_df["id"]))
assert len(official_overlap_posts) == 0
assert len(official_overlap_ids) == 0

test_ds = DreadditDataset(test_df)
final_train_df = pd.concat(
    [fit_df, validation_df], ignore_index=True
)
final_train_ds = DreadditDataset(final_train_df)

test_audit = {
    "rows": int(len(test_df)),
    "columns": int(len(test_df.columns)),
    "unique_posts": int(test_df["post_id"].nunique()),
    "label_0": int((test_df["label"] == 0).sum()),
    "label_1": int((test_df["label"] == 1).sum()),
    "agreement_mean": float(test_df["confidence"].mean()),
    "official_id_overlap": len(official_overlap_ids),
    "official_post_id_overlap": len(official_overlap_posts),
}
with open(OUT_DIR / "official_test_audit.json", "w", encoding="utf-8") as f:
    json.dump(test_audit, f, indent=2)

test_audit

In [ ]:
# ============================================================
# 10. Final test models — hyperparameters are already locked
# ============================================================
if TEST_LOCK_FILE.exists():
    raise RuntimeError(
        "Official test evaluation has already been completed for this RUN_NAME. "
        "Use a new RUN_NAME for any protocol change; do not overwrite the held-out test run."
    )

final_configs = CANDIDATES if RUN_ALL_FINAL_CANDIDATES else [
    next(c for c in CANDIDATES if c["key"] == "M0_Vanilla_BERT"),
    selected_cfg,
]

# Deduplicate in case selected model is M0.
seen, unique_final_configs = set(), []
for cfg in final_configs:
    if cfg["key"] not in seen:
        unique_final_configs.append(cfg)
        seen.add(cfg["key"])

final_rows = []
final_histories = []
selected_checkpoint_dir = OUT_DIR / "selected_model_checkpoint"

for cfg in unique_final_configs:
    result = train_model(
        model_key=cfg["key"],
        train_dataset=final_train_ds,
        evaluation_dataset=test_ds,
        mode=cfg["mode"],
        epsilon=cfg["epsilon"],
        seed=PRIMARY_SEED,
    )

    row = {
        **result["metrics"],
        "mode": cfg["mode"],
        "epsilon": cfg["epsilon"],
        "training_partition": "fit_plus_validation",
        "evaluation_partition": "official_test",
        "preselected_from_validation": cfg["key"] == selected_cfg["key"],
    }
    final_rows.append(row)
    final_histories.append(result["history"])

    pred = result["predictions"].copy()
    pred.insert(0, "id", test_df["id"].values)
    pred.insert(1, "post_id", test_df["post_id"].values)
    pred.insert(2, "text", test_df["text"].values)
    pred.to_csv(
        OUT_DIR / f"test_predictions_{cfg['key']}.csv", index=False
    )
    np.save(
        OUT_DIR / f"test_logits_{cfg['key']}.npy",
        result["logits"],
    )
    result["ece_bins"].to_csv(
        OUT_DIR / f"test_ece_bins_{cfg['key']}.csv", index=False
    )
    result["ace_bins"].to_csv(
        OUT_DIR / f"test_ace_bins_{cfg['key']}.csv", index=False
    )

    if cfg["key"] == selected_cfg["key"]:
        result["model"].save_pretrained(selected_checkpoint_dir)
        tokenizer.save_pretrained(selected_checkpoint_dir)

    release_model(result["model"])

final_test_metrics = pd.DataFrame(final_rows)
final_test_metrics.to_csv(
    OUT_DIR / "official_test_metrics_pre_temperature.csv", index=False
)
pd.concat(final_histories, ignore_index=True).to_csv(
    OUT_DIR / "final_training_history.csv", index=False
)
with open(TEST_LOCK_FILE, "w", encoding="utf-8") as f:
    json.dump({
        "run_name": RUN_NAME,
        "selected_model": selected_cfg["key"],
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "test_file_sha256": input_hashes["dreaddit_test_sha256"],
        "statement": "Official test metrics were generated after validation-only selection.",
    }, f, indent=2)

display(final_test_metrics)

In [ ]:
# ============================================================
# 11. Temperature scaling on untouched calibration data
# ============================================================
selected_model = AutoModelForSequenceClassification.from_pretrained(
    selected_checkpoint_dir
).to(DEVICE)
selected_model.eval()

calibration_logits, calibration_labels = predict_logits(
    selected_model, calibration_ds, seed=PRIMARY_SEED
)
test_logits_selected = np.load(
    OUT_DIR / f"test_logits_{selected_cfg['key']}.npy"
)

class ScalarTemperature(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))

    def forward(self, logits):
        temperature = torch.exp(self.log_temperature).clamp(0.05, 10.0)
        return logits / temperature

    @property
    def value(self):
        return float(torch.exp(self.log_temperature).detach().cpu().item())

scaler = ScalarTemperature().to(DEVICE)
cal_logits_t = torch.tensor(
    calibration_logits, dtype=torch.float32, device=DEVICE
)
cal_labels_t = torch.tensor(
    calibration_labels, dtype=torch.long, device=DEVICE
)
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.LBFGS(
    [scaler.log_temperature],
    lr=0.01,
    max_iter=100,
    line_search_fn="strong_wolfe",
)

def closure():
    optimizer.zero_grad()
    loss = criterion(scaler(cal_logits_t), cal_labels_t)
    loss.backward()
    return loss

with torch.no_grad():
    calibration_nll_before = float(
        criterion(cal_logits_t, cal_labels_t).cpu().item()
    )
optimizer.step(closure)
with torch.no_grad():
    calibration_nll_after = float(
        criterion(scaler(cal_logits_t), cal_labels_t).cpu().item()
    )

temperature = scaler.value

# Calibration-set diagnostics before and after scaling.
cal_probs_before = torch.softmax(cal_logits_t, dim=1).cpu().numpy()
cal_probs_after = torch.softmax(
    scaler(cal_logits_t), dim=1
).detach().cpu().numpy()
cal_metrics_before, _, _, _ = metric_bundle(
    calibration_labels,
    cal_probs_before,
    selected_cfg["key"] + "_calibration_before_TS",
)
cal_metrics_after, _, _, _ = metric_bundle(
    calibration_labels,
    cal_probs_after,
    selected_cfg["key"] + "_calibration_after_TS",
)

# Official test probabilities after the already-fitted scalar T.
test_logits_t = torch.tensor(
    test_logits_selected, dtype=torch.float32, device=DEVICE
)
test_scaled_logits = scaler(test_logits_t).detach().cpu()
test_probs_ts = torch.softmax(test_scaled_logits, dim=1).numpy()

selected_test_pred = pd.read_csv(
    OUT_DIR / f"test_predictions_{selected_cfg['key']}.csv"
)
test_labels = selected_test_pred["true_label"].values.astype(int)

ts_key = f"M3_TS_on_{selected_cfg['key']}"
ts_metrics, ts_ece_bins, ts_ace_bins, ts_predictions = metric_bundle(
    test_labels, test_probs_ts, ts_key
)

# Temperature scaling must not change the predicted class.
assert np.array_equal(
    selected_test_pred["predicted_label"].values,
    ts_predictions["predicted_label"].values,
)

ts_predictions.insert(0, "id", test_df["id"].values)
ts_predictions.insert(1, "post_id", test_df["post_id"].values)
ts_predictions.insert(2, "text", test_df["text"].values)
ts_predictions.to_csv(
    OUT_DIR / "test_predictions_temperature_scaled.csv", index=False
)
ts_ece_bins.to_csv(
    OUT_DIR / "test_ece_bins_temperature_scaled.csv", index=False
)
ts_ace_bins.to_csv(
    OUT_DIR / "test_ace_bins_temperature_scaled.csv", index=False
)

temperature_record = {
    "base_model": selected_cfg["key"],
    "temperature": temperature,
    "fit_partition": "calibration",
    "calibration_nll_before": calibration_nll_before,
    "calibration_nll_after": calibration_nll_after,
    "calibration_metrics_before": cal_metrics_before,
    "calibration_metrics_after": cal_metrics_after,
}
with open(
    OUT_DIR / "temperature_scaling_record.json", "w", encoding="utf-8"
) as f:
    json.dump(temperature_record, f, indent=2)

all_official_test_metrics = pd.concat(
    [final_test_metrics, pd.DataFrame([{
        **ts_metrics,
        "mode": "temperature_scaling",
        "epsilon": selected_cfg["epsilon"],
        "training_partition": "fit_plus_validation",
        "evaluation_partition": "official_test",
        "preselected_from_validation": True,
    }])],
    ignore_index=True,
)
all_official_test_metrics.to_csv(
    OUT_DIR / "official_test_metrics_all.csv", index=False
)

release_model(selected_model)
display(all_official_test_metrics)
temperature_record

In [ ]:
# ============================================================
# 12. HCE, threshold coverage, and selective error
# ============================================================
primary_prediction_files = {
    "M0_Vanilla_BERT": OUT_DIR / "test_predictions_M0_Vanilla_BERT.csv",
    selected_cfg["key"]: OUT_DIR / f"test_predictions_{selected_cfg['key']}.csv",
    ts_key: OUT_DIR / "test_predictions_temperature_scaled.csv",
}

threshold_rows = []
for model_key, path in primary_prediction_files.items():
    pred = pd.read_csv(path)
    wrong = pred["true_label"].values != pred["predicted_label"].values
    confidence = pred["confidence"].values.astype(float)

    for tau in [0.80, 0.90, 0.95]:
        covered = confidence >= tau
        hce = wrong & covered
        threshold_rows.append({
            "model": model_key,
            "threshold": tau,
            "n_total": len(pred),
            "n_errors_total": int(wrong.sum()),
            "n_covered": int(covered.sum()),
            "coverage": float(covered.mean()),
            "hce_count": int(hce.sum()),
            "hce_rate_all": float(hce.mean()),
            "selective_error_rate": (
                float(hce.sum() / covered.sum())
                if covered.sum() > 0 else np.nan
            ),
        })

hce_coverage = pd.DataFrame(threshold_rows)
hce_coverage.to_csv(
    OUT_DIR / "hce_coverage_selective_error.csv", index=False
)
hce_coverage

In [ ]:
# ============================================================
# 13. Reliability diagrams and confidence distributions
# ============================================================
def reliability_figure(model_to_bins, out_path):
    plt.figure(figsize=(7, 7))
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    for model_key, bins in model_to_bins.items():
        sub = bins[bins["count"] > 0]
        plt.plot(
            sub["mean_confidence"],
            sub["accuracy"],
            marker="o",
            label=model_key,
        )
    plt.xlabel("Mean confidence")
    plt.ylabel("Empirical accuracy")
    plt.title("Reliability diagram on the official test set")
    plt.legend(fontsize=8)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.show()
    plt.close()

baseline_bins = pd.read_csv(
    OUT_DIR / "test_ece_bins_M0_Vanilla_BERT.csv"
)
selected_bins = pd.read_csv(
    OUT_DIR / f"test_ece_bins_{selected_cfg['key']}.csv"
)
reliability_figure(
    {
        "Vanilla BERT": baseline_bins,
        selected_cfg["key"]: selected_bins,
    },
    OUT_DIR / "reliability_diagram_baseline_vs_selected.png",
)

plt.figure(figsize=(7, 5))
for model_key, path in {
    "Vanilla BERT": primary_prediction_files["M0_Vanilla_BERT"],
    selected_cfg["key"]: primary_prediction_files[selected_cfg["key"]],
}.items():
    pred = pd.read_csv(path)
    plt.hist(
        pred["confidence"],
        bins=np.linspace(0.5, 1.0, 11),
        alpha=0.45,
        label=model_key,
    )
plt.xlabel("Prediction confidence")
plt.ylabel("Number of test instances")
plt.title("Confidence distribution on the official test set")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(
    OUT_DIR / "confidence_distribution_baseline_vs_selected.png",
    dpi=300,
)
plt.show()
plt.close()

In [ ]:
# ============================================================
# 14. Paired McNemar and paired bootstrap confidence intervals
# ============================================================
baseline_pred = pd.read_csv(
    OUT_DIR / "test_predictions_M0_Vanilla_BERT.csv"
)
selected_pred = pd.read_csv(
    OUT_DIR / f"test_predictions_{selected_cfg['key']}.csv"
)

assert np.array_equal(
    baseline_pred["id"].values, selected_pred["id"].values
)
assert np.array_equal(
    baseline_pred["true_label"].values,
    selected_pred["true_label"].values,
)

base_correct = baseline_pred["correct"].astype(bool).values
sel_correct = selected_pred["correct"].astype(bool).values
b = int((~base_correct & sel_correct).sum())
c = int((base_correct & ~sel_correct).sum())

contingency = [
    [int((base_correct & sel_correct).sum()), c],
    [b, int((~base_correct & ~sel_correct).sum())],
]
mc = mcnemar(contingency, exact=True)
mcnemar_record = pd.DataFrame([{
    "baseline": "M0_Vanilla_BERT",
    "selected": selected_cfg["key"],
    "b_baseline_wrong_selected_correct": b,
    "c_baseline_correct_selected_wrong": c,
    "exact_p_value": float(mc.pvalue),
    "statistic": float(mc.statistic),
}])
mcnemar_record.to_csv(OUT_DIR / "mcnemar_exact.csv", index=False)

def metrics_from_prediction_df(df):
    probs = df[["prob_not_stress", "prob_stress"]].values
    metrics, _, _, _ = metric_bundle(
        df["true_label"].values,
        probs,
        model_key="bootstrap",
    )
    return metrics

observed_base = metrics_from_prediction_df(baseline_pred)
observed_selected = metrics_from_prediction_df(selected_pred)
metric_names = [
    "accuracy", "precision", "recall", "f1_score",
    "brier_score", "nll", "ece", "ace", "hce_0.90_count",
]
observed_delta = {
    m: observed_selected[m] - observed_base[m] for m in metric_names
}

rng = np.random.default_rng(PRIMARY_SEED)
n = len(baseline_pred)
bootstrap_deltas = {m: [] for m in metric_names}

for _ in tqdm(range(BOOTSTRAP_ITERATIONS), desc="Paired bootstrap"):
    idx = rng.integers(0, n, size=n)
    mb = metrics_from_prediction_df(
        baseline_pred.iloc[idx].reset_index(drop=True)
    )
    ms = metrics_from_prediction_df(
        selected_pred.iloc[idx].reset_index(drop=True)
    )
    for m in metric_names:
        bootstrap_deltas[m].append(ms[m] - mb[m])

bootstrap_rows = []
for m in metric_names:
    arr = np.asarray(bootstrap_deltas[m], dtype=float)
    bootstrap_rows.append({
        "metric": m,
        "observed_delta_selected_minus_baseline": observed_delta[m],
        "bootstrap_mean_delta": float(arr.mean()),
        "ci_2.5": float(np.percentile(arr, 2.5)),
        "ci_97.5": float(np.percentile(arr, 97.5)),
        "iterations": BOOTSTRAP_ITERATIONS,
    })

bootstrap_table = pd.DataFrame(bootstrap_rows)
bootstrap_table.to_csv(
    OUT_DIR / "paired_bootstrap_95ci.csv", index=False
)
display(mcnemar_record)
bootstrap_table

In [ ]:
# ============================================================
# 15. Repeated-seed paired evaluation
# ============================================================
if RUN_REPEATED_SEEDS:
    repeated_rows = []

    repeated_configs = [
        next(c for c in CANDIDATES if c["key"] == "M0_Vanilla_BERT"),
        selected_cfg,
    ]
    # Deduplicate if M0 is selected.
    repeated_unique = []
    seen = set()
    for cfg in repeated_configs:
        if cfg["key"] not in seen:
            repeated_unique.append(cfg)
            seen.add(cfg["key"])

    for seed in REPEATED_SEEDS:
        for cfg in repeated_unique:
            # train_model resets the same seed immediately before every model.
            result = train_model(
                model_key=cfg["key"],
                train_dataset=final_train_ds,
                evaluation_dataset=test_ds,
                mode=cfg["mode"],
                epsilon=cfg["epsilon"],
                seed=seed,
            )
            repeated_rows.append({
                "seed": seed,
                "model": cfg["key"],
                **result["metrics"],
            })

            pred = result["predictions"].copy()
            pred.insert(0, "id", test_df["id"].values)
            pred.to_csv(
                OUT_DIR
                / f"repeated_seed_{seed}_predictions_{cfg['key']}.csv",
                index=False,
            )
            release_model(result["model"])

    repeated_raw = pd.DataFrame(repeated_rows)
    repeated_raw.to_csv(
        OUT_DIR / "repeated_seed_raw_metrics.csv", index=False
    )

    repeated_summary = (
        repeated_raw
        .groupby("model")[
            [
                "accuracy", "precision", "recall", "f1_score",
                "brier_score", "nll", "ece", "ace",
                "hce_0.90_count", "coverage_0.90",
            ]
        ]
        .agg(["mean", "std"])
    )
    repeated_summary.to_csv(
        OUT_DIR / "repeated_seed_mean_std.csv"
    )
    display(repeated_raw)
    display(repeated_summary)
else:
    print("Repeated-seed evaluation disabled.")

In [ ]:
# ============================================================
# 16. Official split duplicate-text audit and sensitivity analysis
# ============================================================
def normalize_text(text):
    return " ".join(str(text).strip().lower().split())

train_text_map = {}
for _, row in train_full.iterrows():
    key = normalize_text(row["text"])
    train_text_map.setdefault(key, []).append(row)

test_norm = test_df["text"].map(normalize_text)
overlap_keys = set(train_text_map).intersection(set(test_norm))

overlap_rows = []
overlap_test_indices = []
for key in sorted(overlap_keys):
    for row in train_text_map[key]:
        overlap_rows.append({
            "source": "official_train",
            "normalized_text_sha256": hashlib.sha256(
                key.encode("utf-8")
            ).hexdigest(),
            "id": row["id"],
            "post_id": row["post_id"],
            "label": int(row["label"]),
            "confidence": float(row["confidence"]),
            "text": row["text"],
        })
    matching = test_df.loc[test_norm == key]
    overlap_test_indices.extend(matching.index.tolist())
    for _, row in matching.iterrows():
        overlap_rows.append({
            "source": "official_test",
            "normalized_text_sha256": hashlib.sha256(
                key.encode("utf-8")
            ).hexdigest(),
            "id": row["id"],
            "post_id": row["post_id"],
            "label": int(row["label"]),
            "confidence": float(row["confidence"]),
            "text": row["text"],
        })

overlap_table = pd.DataFrame(overlap_rows)
overlap_table.to_csv(
    OUT_DIR / "official_split_exact_text_overlap.csv", index=False
)

sensitivity_rows = []
keep_mask = ~test_df.index.isin(overlap_test_indices)
for model_key, path in primary_prediction_files.items():
    pred = pd.read_csv(path)
    metrics_all = metrics_from_prediction_df(pred)
    metrics_excluding = metrics_from_prediction_df(
        pred.loc[keep_mask].reset_index(drop=True)
    )
    for metric in [
        "accuracy", "precision", "recall", "f1_score",
        "brier_score", "nll", "ece", "ace", "hce_0.90_count",
    ]:
        sensitivity_rows.append({
            "model": model_key,
            "metric": metric,
            "official_full_test": metrics_all[metric],
            "excluding_exact_text_overlap": metrics_excluding[metric],
            "difference_excluding_minus_full": (
                metrics_excluding[metric] - metrics_all[metric]
            ),
            "excluded_test_instances": len(overlap_test_indices),
        })

duplicate_sensitivity = pd.DataFrame(sensitivity_rows)
duplicate_sensitivity.to_csv(
    OUT_DIR / "duplicate_overlap_sensitivity.csv", index=False
)
print("Exact normalized text overlaps:", len(overlap_keys))
print("Excluded test instances in sensitivity analysis:", len(overlap_test_indices))
duplicate_sensitivity

In [ ]:
# ============================================================
# 17. Top2Vec topic partition and topic-aware calibration
# ============================================================
# This is a post-prediction diagnostic. It is fitted only to official
# test texts and does not influence classifier training or selection.
# There is deliberately no fallback to a different topic model.

if RUN_TOP2VEC:
    try:
        from top2vec import Top2Vec
    except Exception as exc:
        raise RuntimeError(
            "Top2Vec is required. No fallback method is permitted in "
            "the canonical experiment."
        ) from exc

    set_all_seeds(PRIMARY_SEED)
    topic_model = Top2Vec(
        test_df["text"].astype(str).tolist(),
        speed="learn",
        workers=1,
    )
    num_topics = int(topic_model.get_num_topics())
    doc_ids = list(range(len(test_df)))
    topic_nums, topic_scores, _, _ = topic_model.get_documents_topics(
        doc_ids=doc_ids, num_topics=1
    )
    topic_assignment = np.asarray(topic_nums).reshape(-1).astype(int)
    topic_score = np.asarray(topic_scores).reshape(-1).astype(float)

    topic_words, word_scores, topic_ids = topic_model.get_topics()
    keyword_rows = []
    for topic_id, words in zip(topic_ids, topic_words):
        keyword_rows.append({
            "topic": int(topic_id),
            "keywords": ", ".join([str(w) for w in words[:10]]),
        })
    topic_keywords = pd.DataFrame(keyword_rows)
    topic_keywords.to_csv(
        OUT_DIR / "top2vec_topic_keywords.csv", index=False
    )

    try:
        topic_model.save(str(OUT_DIR / "top2vec_canonical_model"))
        topic_model_saved = True
    except Exception as exc:
        topic_model_saved = False
        (OUT_DIR / "top2vec_save_warning.txt").write_text(
            repr(exc), encoding="utf-8"
        )

    diagnostic_pred = selected_pred.copy()
    diagnostic_pred["topic"] = topic_assignment
    diagnostic_pred["topic_score"] = topic_score
    diagnostic_pred.to_csv(
        OUT_DIR / "selected_predictions_with_topic.csv", index=False
    )

    def local_adaptive_ece(group):
        n = len(group)
        # At least ~10 samples per bin; one bin for groups below 20.
        n_bins = max(1, min(10, n // 10))
        probs = group[["prob_not_stress", "prob_stress"]].values
        y_true = group["true_label"].values.astype(int)
        y_pred = probs.argmax(axis=1)
        conf = probs.max(axis=1)
        value, bins = adaptive_calibration_error(
            y_true, y_pred, conf, n_bins=n_bins
        )
        return value, n_bins

    topic_rows = []
    for topic, group in diagnostic_pred.groupby("topic"):
        local_ece, n_bins = local_adaptive_ece(group)
        topic_rows.append({
            "topic": int(topic),
            "count": int(len(group)),
            "accuracy": float(group["correct"].mean()),
            "mean_confidence": float(group["confidence"].mean()),
            "local_adaptive_ece": float(local_ece),
            "adaptive_bins": int(n_bins),
            "weighted_tace_contribution": float(
                len(group) / len(diagnostic_pred) * local_ece
            ),
            "interpretation_flag": (
                "small_topic_n_lt_30" if len(group) < 30 else ""
            ),
        })

    topic_table = pd.DataFrame(topic_rows).merge(
        topic_keywords, on="topic", how="left"
    )
    TACE = float(topic_table["weighted_tace_contribution"].sum())

    # Conditional bootstrap: topic assignments are held fixed.
    rng = np.random.default_rng(PRIMARY_SEED)
    topic_groups = {
        int(t): g.reset_index(drop=True)
        for t, g in diagnostic_pred.groupby("topic")
    }
    original_weights = {
        t: len(g) / len(diagnostic_pred) for t, g in topic_groups.items()
    }

    topic_boot_values = {t: [] for t in topic_groups}
    tace_boot_values = []

    for _ in tqdm(
        range(TOPIC_BOOTSTRAP_ITERATIONS),
        desc="Topic calibration bootstrap",
    ):
        weighted_total = 0.0
        for topic, group in topic_groups.items():
            idx = rng.integers(0, len(group), size=len(group))
            sample = group.iloc[idx].reset_index(drop=True)
            value, _ = local_adaptive_ece(sample)
            topic_boot_values[topic].append(value)
            weighted_total += original_weights[topic] * value
        tace_boot_values.append(weighted_total)

    ci_rows = []
    for topic, values in topic_boot_values.items():
        arr = np.asarray(values, dtype=float)
        ci_rows.append({
            "topic": topic,
            "local_ece_ci_2.5": float(np.percentile(arr, 2.5)),
            "local_ece_ci_97.5": float(np.percentile(arr, 97.5)),
        })
    topic_ci = pd.DataFrame(ci_rows)
    topic_table = topic_table.merge(topic_ci, on="topic", how="left")
    topic_table = topic_table.sort_values(
        "local_adaptive_ece", ascending=False
    )
    topic_table.to_csv(
        OUT_DIR / "topic_aware_calibration_table.csv", index=False
    )

    tace_arr = np.asarray(tace_boot_values, dtype=float)
    tace_record = {
        "diagnostic_model": selected_cfg["key"],
        "topic_model": "Top2Vec",
        "topic_corpus": "official_test_text_only",
        "num_topics": num_topics,
        "tace": TACE,
        "tace_ci_2.5": float(np.percentile(tace_arr, 2.5)),
        "tace_ci_97.5": float(np.percentile(tace_arr, 97.5)),
        "bootstrap_iterations": TOPIC_BOOTSTRAP_ITERATIONS,
        "topic_assignments_fixed_in_bootstrap": True,
        "top2vec_model_saved": topic_model_saved,
        "comparison_metric": "global ACE uses equal-frequency bins",
    }
    with open(
        OUT_DIR / "tace_summary.json", "w", encoding="utf-8"
    ) as f:
        json.dump(tace_record, f, indent=2)

    display(topic_table.head(10))
    tace_record
else:
    print("Top2Vec disabled.")

In [ ]:
# ============================================================
# 18. Final verified summary, checksums, and archive
# ============================================================
def collect_output_hashes(directory):
    rows = []
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            rows.append({
                "relative_path": str(path.relative_to(directory)),
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            })
    return pd.DataFrame(rows)

primary_summary = {
    "selected_configuration": selection_record,
    "official_test_metrics_file": "official_test_metrics_all.csv",
    "primary_comparison": {
        "baseline": "M0_Vanilla_BERT",
        "selected_training_time_model": selected_cfg["key"],
    },
    "temperature_scaling": temperature_record,
    "test_results_not_used_for_selection": True,
    "input_hashes": input_hashes,
    "split_assignment_file": "development_split_assignment.csv",
    "split_assignment_sha256": sha256_file(
        OUT_DIR / "development_split_assignment.csv"
    ),
    "completed_utc": datetime.now(timezone.utc).isoformat(),
}
with open(
    OUT_DIR / "verified_experiment_summary.json", "w", encoding="utf-8"
) as f:
    json.dump(primary_summary, f, indent=2)

hash_table = collect_output_hashes(OUT_DIR)
hash_table.to_csv(OUT_DIR / "output_file_hashes.csv", index=False)

# Hash table changed after it was written, so also hash the final table itself.
final_manifest = {
    "run_name": RUN_NAME,
    "input_hashes": input_hashes,
    "selected_model": selected_cfg["key"],
    "output_hash_table_sha256": sha256_file(
        OUT_DIR / "output_file_hashes.csv"
    ),
    "files_before_archive": int(len(hash_table)),
}
with open(OUT_DIR / "MANIFEST.json", "w", encoding="utf-8") as f:
    json.dump(final_manifest, f, indent=2)

archive_path = shutil.make_archive(
    str(ROOT / RUN_NAME), "zip", root_dir=OUT_DIR
)
print("Canonical output archive:", archive_path)
print("Do not revise the manuscript until this archive has been preserved.")

In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil
import os

drive.mount("/content/drive", force_remount=False)

source_zip = Path("/content/calibbert_jucs_canonical_v1.zip")
drive_folder = Path("/content/drive/MyDrive/CalibBERT_JUCS_Final_Outputs")
drive_folder.mkdir(parents=True, exist_ok=True)

if source_zip.exists():
    destination = drive_folder / source_zip.name
    shutil.copy2(source_zip, destination)

    source_size = source_zip.stat().st_size
    destination_size = destination.stat().st_size

    if source_size != destination_size:
        raise RuntimeError("Ukuran file hasil salinan tidak sama.")

    print("ZIP berhasil disimpan permanen di:")
    print(destination)
    print(f"Ukuran: {destination_size / (1024**2):.2f} MB")

    files.download(str(source_zip))
else:
    raise FileNotFoundError(
        "ZIP belum ditemukan. Pastikan seluruh sel eksperimen dan sel packaging "
        "sudah selesai dijalankan."
    )